In [1]:
# Import required libraries
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report
from textblob import TextBlob

In [2]:
# Download stopwords if not already available
nltk.download('stopwords')

# Step 1: Load Dataset
file_path = "Blogs.csv"  # Update this path if needed
df = pd.read_csv(file_path, encoding="ISO-8859-1")

# Display dataset structure
df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sohebkhan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,Data,Labels
0,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,alt.atheism
1,Newsgroups: alt.atheism\nPath: cantaloupe.srv....,alt.atheism
2,Path: cantaloupe.srv.cs.cmu.edu!das-news.harva...,alt.atheism
3,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,alt.atheism
4,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:53...,alt.atheism


In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sohebkhan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
# Step 2: Text Preprocessing Function
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\[.*?\]', '', text)  # Remove square brackets
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'\w*\d\w*', '', text)  # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = " ".join([word for word in text.split() if word not in stopwords.words('english')])  # Remove stopwords
    return text

In [5]:
# Apply text preprocessing
df['Cleaned_Data'] = df['Data'].astype(str).apply(clean_text)


In [7]:
# Step 3: Feature Extraction (TF-IDF)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['Cleaned_Data'])
y = df['Labels']

In [8]:
# Step 4: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [9]:
# Step 5: Train Naïve Bayes Classifier
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [10]:
# Step 6: Make Predictions
y_pred = model.predict(X_test)

In [11]:
# Step 7: Model Evaluation
print("\nModel Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))



Model Accuracy: 0.8525

Classification Report:
                           precision    recall  f1-score   support

             alt.atheism       0.52      0.83      0.64        18
           comp.graphics       0.79      0.83      0.81        18
 comp.os.ms-windows.misc       0.95      0.91      0.93        22
comp.sys.ibm.pc.hardware       0.88      0.84      0.86        25
   comp.sys.mac.hardware       0.86      0.86      0.86        21
          comp.windows.x       1.00      0.84      0.91        25
            misc.forsale       1.00      0.78      0.88        18
               rec.autos       0.90      1.00      0.95        18
         rec.motorcycles       1.00      0.88      0.93        16
      rec.sport.baseball       0.88      0.83      0.86        18
        rec.sport.hockey       0.83      1.00      0.91        15
               sci.crypt       0.76      1.00      0.86        19
         sci.electronics       0.79      0.94      0.86        16
                 sci.med  

In [12]:
# Step 8: Sentiment Analysis Function
def get_sentiment(text):
    analysis = TextBlob(text)
    if analysis.sentiment.polarity > 0:
        return "Positive"
    elif analysis.sentiment.polarity < 0:
        return "Negative"
    else:
        return "Neutral"

In [13]:
# Apply sentiment analysis to blog posts
df['Sentiment'] = df['Data'].astype(str).apply(get_sentiment)


In [14]:
# Step 9: Sentiment Distribution by Category
sentiment_distribution = df.groupby('Labels')['Sentiment'].value_counts().unstack()
print("\nSentiment Distribution Across Categories:\n", sentiment_distribution)



Sentiment Distribution Across Categories:
 Sentiment                 Negative  Positive
Labels                                      
alt.atheism                     23        77
comp.graphics                   24        76
comp.os.ms-windows.misc         22        78
comp.sys.ibm.pc.hardware        20        80
comp.sys.mac.hardware           24        76
comp.windows.x                  27        73
misc.forsale                    16        84
rec.autos                       17        83
rec.motorcycles                 26        74
rec.sport.baseball              29        71
rec.sport.hockey                34        66
sci.crypt                       19        81
sci.electronics                 19        81
sci.med                         29        71
sci.space                       27        73
soc.religion.christian          13        87
talk.politics.guns              30        70
talk.politics.mideast           22        78
talk.politics.misc              22        78
talk.religi